In [1]:
import pandas as pd
from pandas import DataFrame

In [2]:
#item_categories_df = pd.read_csv("../data/item_categories.csv")
items_df = pd.read_csv('../data/items.csv')
sales_train_df = pd.read_csv("../data/sales_train.csv")
shops_df = pd.read_csv("../data/shops.csv")
test_df = pd.read_csv("../data/test.csv")

In [3]:
test_df

,ID,shop_id,item_id
0,0,5,5037
1,1,5,5320
2,2,5,5233
3,3,5,5232
4,4,5,5268
...,...,...,...
214195,214195,45,18454
214196,214196,45,16188
214197,214197,45,15757
214198,214198,45,19648


In [4]:
len(sales_train_df)

2935849

In [5]:
def delete_equal_snop_name(sales_df: DataFrame, shop_df: DataFrame, test_df: DataFrame) -> DataFrame:
    # "Жуковский ул. Чкалова 39м?" and "Жуковский ул. Чкалова 39м²" (correct_id=11, incorrect=10)
    # "Якутск Орджоникидзе, 56" and "!Якутск Орджоникидзе, 56 фран"" (correct_id=57, incorrect=0)
    # "Якутск ТЦ "Центральный"" and "!Якутск ТЦ "Центральный" фран" (correct_id=58, incorrect=1)
    
    equal_shop_map = {
        10: 11,
        0: 57,
        1: 58,
    }
    shop_df = shop_df.copy()
    shop_df['new_shop_id'] = shop_df['shop_id']
    
    # replace equal
    shop_df['new_shop_id'] = shop_df['new_shop_id'].replace(equal_shop_map)
    sales_df['shop_id'] = sales_df['shop_id'].replace(equal_shop_map)
    test_df['shop_id'] = test_df['shop_id'].replace(equal_shop_map)

    # remove equal
    shop_df = shop_df[shop_df['shop_id'] == shop_df['new_shop_id']]

    # drop help column and reset index
    shop_df = shop_df.drop(['new_shop_id'], axis=1)
    shop_df = shop_df.reset_index(drop=True)

    old_new_id_map = dict(zip(shop_df['shop_id'], shop_df.index))

    # replace old ids with new after .reset_index
    sales_df['shop_id'] = sales_df['shop_id'].replace(old_new_id_map)
    test_df['shop_id'] = test_df['shop_id'].replace(old_new_id_map)
    shop_df['shop_id'] = shop_df.index

    
    return sales_df, shop_df, test_df

In [6]:
def remove_outliers(sales_df: DataFrame) -> DataFrame:
    median = sales_df['item_cnt_month'].median()
    sales_without_ones = sales_df[sales_df['item_cnt_month'] != 1]
    MAD = (sales_without_ones['item_cnt_month'] - median).abs().mean() * 1.4826

    sales_df = sales_df[sales_df['item_cnt_month'].between(median - 3*MAD, median + 3*MAD)]
    
    print("MAD: ", median - 3*MAD, median + 3*MAD)
    return sales_df

In [7]:
def move_to_month(sales_df: DataFrame) -> DataFrame:
    sales_df = sales_df.groupby(['date_block_num', 'shop_id', 'item_id']).agg({
        'item_cnt_day': 'sum',
        'item_price': 'first',
        'item_category_id': 'first'
    }).reset_index()
    sales_df = sales_df.drop('item_price', axis=1)
    sales_df.rename(columns={'item_cnt_day': 'item_cnt_month'}, inplace=True)
    
    return sales_df

In [8]:
def fix_data(
    sales_df: DataFrame, 
    items_df: DataFrame, 
    shop_df: DataFrame, 
    test_df: DataFrame,
) -> tuple[DataFrame,]:
    # Check negative price
    # TODO не удалять а заменить на среднюю цену по предмету или нет, у нас 3 млн строк)
    negative_mask = sales_df['item_price'] <= 0
    negative_count = negative_mask.sum()
    if (negative_count > 0):
        print(f"Negative price were find ({negative_count})")
        sales_df = sales_df[~negative_mask]

    # equal shop names
    sales_df, shop_df, test_df = delete_equal_snop_name(sales_df, shop_df, test_df)
    
    # Add category id in data
    sales_df = sales_df.join(items_df['item_category_id'], on='item_id')
    test_df = test_df.join(items_df['item_category_id'], on='item_id')

    # Move to cnt_month
    sales_df = move_to_month(sales_df)

    # Remove outliers
    sales_df = remove_outliers(sales_df)
    
    return sales_df, shop_df, test_df


In [9]:
sales_train_df_fix, shops_df_fix, test_df_fix = fix_data(sales_train_df,
                                                         items_df,
                                                         shops_df,
                                                         test_df,)

Negative price were find (1)
MAD:  -15.553435548674447 17.553435548674447


In [10]:
# Потери в данных
1 - len(sales_train_df_fix)/len(move_to_month(sales_train_df.join(items_df['item_category_id'], on='item_id')))

0.009259075124104843

In [11]:
sales_train_df_fix.head(3)

,date_block_num,shop_id,item_id,item_cnt_month,item_category_id
0,0,0,27,1.0,19
1,0,0,33,1.0,37
2,0,0,317,1.0,45


In [12]:
shops_df_fix.head(3)

,shop_name,shop_id
0,"Адыгея ТЦ ""Мега""",0
1,"Балашиха ТРК ""Октябрь-Киномир""",1
2,"Волжский ТЦ ""Волга Молл""",2


In [13]:
test_df_fix.head()

,ID,shop_id,item_id,item_category_id
0,0,3,5037,19
1,1,3,5320,55
2,2,3,5233,19
3,3,3,5232,23
4,4,3,5268,20


In [14]:
sales_train_df_fix.to_csv("../data/sales_train_preprocessed.csv")
shops_df_fix.to_csv("../data/shops_preprocessed.csv")
test_df_fix.to_csv("../data/test_preprocessed.csv")